# 24.8 Neuroevolution (NEAT)

NEAT evolves neural weights and topology while protecting new structural innovations.

Neuroevolution applies evolutionary search to neural networks. NEAT is the classic method that evolves not only weights, but also the network graph itself.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 2406
rng = np.random.default_rng(SEED)


def sphere_loss(x):
    x = np.asarray(x, dtype=float)
    return float(np.sum(x * x))


def rastrigin_loss(x):
    x = np.asarray(x, dtype=float)
    n = x.size
    waves = x * x - 10.0 * np.cos(2.0 * np.pi * x)
    return float(10.0 * n + np.sum(waves))


def constrained_loss(x):
    x = np.asarray(x, dtype=float)
    base = sphere_loss(x - np.array([1.0, -1.0]))
    violation = max(0.0, x[0] + x[1] - 1.0)
    lower = np.maximum(0.0, -5.0 - x)
    upper = np.maximum(0.0, x - 5.0)
    bounds_penalty = float(np.sum(lower * lower + upper * upper))
    return float(base + 50.0 * violation * violation + 20.0 * bounds_penalty)


def make_black_box_ladder():
    ladder = []
    ladder.append({
        "name": "D1 1-D quadratic",
        "dim": 1,
        "bounds": np.array([[-5.0, 5.0]]),
        "objective": lambda x: float((np.asarray(x)[0] - 3.0) ** 2),
        "optimum": np.array([3.0]),
    })
    ladder.append({
        "name": "D2 2-D sphere",
        "dim": 2,
        "bounds": np.array([[-5.0, 5.0], [-5.0, 5.0]]),
        "objective": sphere_loss,
        "optimum": np.zeros(2),
    })
    ladder.append({
        "name": "D3 multimodal Rastrigin",
        "dim": 2,
        "bounds": np.array([[-5.12, 5.12], [-5.12, 5.12]]),
        "objective": rastrigin_loss,
        "optimum": np.zeros(2),
    })
    ladder.append({
        "name": "D4 constrained penalty",
        "dim": 2,
        "bounds": np.array([[-5.0, 5.0], [-5.0, 5.0]]),
        "objective": constrained_loss,
        "optimum": np.array([1.0, -1.0]),
    })
    ladder.append({
        "name": "D5 high-dim Rastrigin",
        "dim": 30,
        "bounds": np.tile(np.array([[-5.12, 5.12]]), (30, 1)),
        "objective": rastrigin_loss,
        "optimum": np.zeros(30),
    })
    return ladder


def sample_uniform(bounds, count, generator):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    samples = generator.uniform(lows, highs, size=(count, len(bounds)))
    return samples


def clip_to_bounds(x, bounds):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    return np.minimum(np.maximum(x, lows), highs)


def reflect_to_bounds(x, v, bounds):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    below = x < lows
    above = x > highs
    x = np.minimum(np.maximum(x, lows), highs)
    v = np.where(below | above, -0.5 * v, v)
    return x, v


def plot_landscape(ax, rung):
    dim = rung["dim"]
    bounds = rung["bounds"]
    if dim == 1:
        xs = np.linspace(bounds[0, 0], bounds[0, 1], 200)
        ys = np.array([rung["objective"]([x]) for x in xs])
        ax.plot(xs, ys, color="0.35")
        ax.set_xlabel("x")
        ax.set_ylabel("loss")
        return
    grid_x = np.linspace(bounds[0, 0], bounds[0, 1], 80)
    grid_y = np.linspace(bounds[1, 0], bounds[1, 1], 80)
    xx, yy = np.meshgrid(grid_x, grid_y)
    zz = np.zeros_like(xx)
    for row in range(xx.shape[0]):
        for col in range(xx.shape[1]):
            probe = np.zeros(dim)
            probe[0] = xx[row, col]
            probe[1] = yy[row, col]
            zz[row, col] = rung["objective"](probe)
    ax.contourf(xx, yy, zz, levels=24, cmap="viridis")
    ax.set_xlabel("x0")
    ax.set_ylabel("x1")


def make_regression_task(rung):
    generator = np.random.default_rng(2480 + rung["dim"])
    dim = rung["dim"]
    count = 80 if dim <= 2 else 140
    x = sample_uniform(rung["bounds"], count, generator)
    y = np.array([-rung["objective"](row) for row in x])
    y = (y - np.mean(y)) / (np.std(y) + 1.0e-9)
    return x, y


def initial_genome(dim, hidden_count, generator, innovation_start=1):
    genes = []
    innovation = innovation_start
    for input_idx in range(dim):
        genes.append({"src": input_idx, "dst": dim, "w": float(generator.normal(0.0, 0.5)), "enabled": True, "innov": innovation})
        innovation += 1
    for hidden_idx in range(hidden_count):
        node_id = dim + 1 + hidden_idx
        source = int(generator.integers(0, dim))
        genes.append({"src": source, "dst": node_id, "w": float(generator.normal(0.0, 0.5)), "enabled": True, "innov": innovation})
        innovation += 1
        genes.append({"src": node_id, "dst": dim, "w": float(generator.normal(0.0, 0.5)), "enabled": True, "innov": innovation})
        innovation += 1
    return {"dim": dim, "hidden": hidden_count, "genes": genes, "next_node": dim + 1 + hidden_count}


def copy_genome(genome):
    return {"dim": genome["dim"], "hidden": genome["hidden"], "next_node": genome["next_node"], "genes": [dict(g) for g in genome["genes"]]}


def evaluate_genome(genome, x):
    outputs = []
    output_node = genome["dim"]
    for row in x:
        values = {idx: float(row[idx]) for idx in range(genome["dim"])}
        hidden_nodes = sorted({g["dst"] for g in genome["genes"] if g["dst"] != output_node})
        for node in hidden_nodes:
            incoming = [g for g in genome["genes"] if g["dst"] == node and g["enabled"]]
            total = 0.0
            for gene in incoming:
                total += values.get(gene["src"], 0.0) * gene["w"]
            values[node] = np.tanh(total)
        total = 0.0
        for gene in genome["genes"]:
            if gene["dst"] == output_node and gene["enabled"]:
                total += values.get(gene["src"], 0.0) * gene["w"]
        outputs.append(total)
    return np.array(outputs)


def genome_fitness(genome, x, y, topology_penalty=0.002):
    pred = evaluate_genome(genome, x)
    mse = float(np.mean((pred - y) ** 2))
    enabled = sum(1 for g in genome["genes"] if g["enabled"])
    return -mse - topology_penalty * enabled


def mutate_genome(genome, generator, innovation_counter, add_node_rate=0.08, add_conn_rate=0.12):
    child = copy_genome(genome)
    for gene in child["genes"]:
        if generator.random() < 0.9:
            gene["w"] += float(generator.normal(0.0, 0.25))
    if generator.random() < add_conn_rate:
        possible_sources = list(range(child["dim"])) + list(range(child["dim"] + 1, child["next_node"]))
        source = int(generator.choice(possible_sources))
        dest = child["dim"]
        child["genes"].append({"src": source, "dst": dest, "w": float(generator.normal(0.0, 0.5)), "enabled": True, "innov": innovation_counter})
        innovation_counter += 1
    if child["genes"] and generator.random() < add_node_rate:
        gene_index = int(generator.integers(0, len(child["genes"])))
        split_gene = child["genes"][gene_index]
        if split_gene["enabled"]:
            split_gene["enabled"] = False
            new_node = child["next_node"]
            child["next_node"] += 1
            child["hidden"] += 1
            child["genes"].append({"src": split_gene["src"], "dst": new_node, "w": 1.0, "enabled": True, "innov": innovation_counter})
            innovation_counter += 1
            child["genes"].append({"src": new_node, "dst": split_gene["dst"], "w": split_gene["w"], "enabled": True, "innov": innovation_counter})
            innovation_counter += 1
    return child, innovation_counter


def compatibility_distance(a, b, c1=1.0, c2=1.0, c3=0.4):
    genes_a = {g["innov"]: g for g in a["genes"]}
    genes_b = {g["innov"]: g for g in b["genes"]}
    all_ids = sorted(set(genes_a) | set(genes_b))
    matching = [idx for idx in all_ids if idx in genes_a and idx in genes_b]
    disjoint = [idx for idx in all_ids if idx not in genes_a or idx not in genes_b]
    max_a = max(genes_a) if genes_a else 0
    max_b = max(genes_b) if genes_b else 0
    excess = [idx for idx in disjoint if idx > min(max_a, max_b)]
    disjoint_only = [idx for idx in disjoint if idx <= min(max_a, max_b)]
    n = max(1, max(len(genes_a), len(genes_b)))
    if matching:
        weight_gap = np.mean([abs(genes_a[idx]["w"] - genes_b[idx]["w"]) for idx in matching])
    else:
        weight_gap = 0.0
    delta = c1 * len(excess) / n + c2 * len(disjoint_only) / n + c3 * weight_gap
    return float(delta)


def speciate(population, threshold=1.0):
    species = []
    representatives = []
    for genome in population:
        placed = False
        for idx, rep in enumerate(representatives):
            if compatibility_distance(genome, rep) < threshold:
                species[idx].append(genome)
                placed = True
                break
        if not placed:
            representatives.append(genome)
            species.append([genome])
    return species




## The concept, built once: innovations and compatibility

NEAT protects structural innovations with compatibility distance $$\delta=\frac{c_1E}{N}+\frac{c_2D}{N}+c_3\bar W.$$ We first reproduce the lesson's prediction, mutation error, and distance calculation.

In [ ]:
x = np.array([2.0, 1.0])
y = 1.0
w = np.array([1.0, -0.5])
pred = float(np.dot(w, x))
err = (pred - y) ** 2
mutated_w = np.array([1.2, -0.4])
mutated_pred = float(np.dot(mutated_w, x))
mutated_err = (mutated_pred - y) ** 2
assert pred == 1.5
assert err == 0.25
assert mutated_pred == 2.0
assert mutated_err == 1.0
genome_a = {"dim": 2, "hidden": 0, "next_node": 3, "genes": [{"innov": 1, "w": 0.5}, {"innov": 2, "w": -1.0}, {"innov": 4, "w": 0.7}]}
genome_b = {"dim": 2, "hidden": 0, "next_node": 3, "genes": [{"innov": 1, "w": 0.6}, {"innov": 3, "w": 1.2}, {"innov": 4, "w": 0.4}]}
delta = compatibility_distance(genome_a, genome_b)
assert round(delta, 3) == 0.747
print("prediction/error", pred, err)
print("mutated prediction/error", mutated_pred, mutated_err)
print("compatibility", round(delta, 3))

Now we implement a tiny NEAT-style optimizer with weight mutation, add-connection and add-node mutation, innovation-number alignment for distance, species grouping, and fitness sharing.

In [ ]:
def neat_like_optimize(x, y, population_size=28, generations=35, speciation=True, seed=0):
    generator = np.random.default_rng(seed)
    dim = x.shape[1]
    population = [initial_genome(dim, 0, generator, innovation_start=1 + idx * (dim + 2)) for idx in range(population_size)]
    innovation_counter = 10000
    best = None
    best_fit = -np.inf
    history = []
    species_counts = []
    for generation in range(generations):
        fitness = np.array([genome_fitness(genome, x, y) for genome in population])
        if speciation:
            species = speciate(population, threshold=1.2)
            shared = fitness.copy()
            for group in species:
                divisor = max(1, len(group))
                group_ids = {id(genome) for genome in group}
                for idx, genome in enumerate(population):
                    if id(genome) in group_ids:
                        shared[idx] = shared[idx] / divisor
            selection_scores = shared
            species_counts.append(len(species))
        else:
            selection_scores = fitness
            species_counts.append(1)
        top_idx = int(np.argmax(fitness))
        if fitness[top_idx] > best_fit:
            best_fit = float(fitness[top_idx])
            best = copy_genome(population[top_idx])
        order = np.argsort(selection_scores)[::-1]
        elites = [copy_genome(population[idx]) for idx in order[: max(2, population_size // 5)]]
        children = elites[:2]
        while len(children) < population_size:
            parent = copy_genome(elites[int(generator.integers(0, len(elites)))])
            child, innovation_counter = mutate_genome(parent, generator, innovation_counter)
            children.append(child)
        population = children
        history.append(best_fit)
    return {"best_genome": best, "best_fitness": best_fit, "history": np.array(history), "species_counts": np.array(species_counts), "population": population}


## The dataset ladder

We use the F4 black-box objective ladder inline: D1 is 1-D, D2 is a sphere, D3 is multimodal, D4 adds a constraint penalty, and D5 is high-dimensional.

In [ ]:
ladder = make_black_box_ladder()
for rung in ladder:
    preview = sample_uniform(rung["bounds"], 3, rng)
    values = [rung["objective"](row) for row in preview]
    print(rung["name"], "dim=", rung["dim"], "bounds_shape=", rung["bounds"].shape)
    print("sample", np.round(preview[:2], 3))
    print("loss", np.round(values[:2], 3))

In [ ]:
neat_results = []
for idx, rung in enumerate(ladder):
    x_task, y_task = make_regression_task(rung)
    result = neat_like_optimize(x_task, y_task, seed=SEED + idx)
    neat_results.append((result, x_task, y_task))
    print(rung["name"], "best_fitness=", round(result["best_fitness"], 4), "convergence_gen=", int(np.argmax(result["history"])), "species_last=", int(result["species_counts"][-1]))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for idx, rung in enumerate(ladder):
    ax = axes[0, idx]
    plot_landscape(ax, rung)
    result, x_task, y_task = neat_results[idx]
    best = result["best_genome"]
    pts = x_task[:30]
    if rung["dim"] == 1:
        pred = evaluate_genome(best, pts)
        ax.scatter(pts[:, 0], pred, s=18, color="tab:red")
    else:
        ax.scatter(pts[:, 0], pts[:, 1], s=18, color="tab:red")
    ax.set_title(rung["name"] + f"\n{len(best['genes'])} genes")
summary_ax = axes[1, 0]
for idx, item in enumerate(neat_results):
    summary_ax.plot(item[0]["history"], label=f"D{idx + 1}")
summary_ax.set_title("best fitness vs generation")
summary_ax.set_xlabel("generation")
summary_ax.set_ylabel("fitness")
summary_ax.legend()
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on D5: removing speciation

New structures often start weak. Without compatibility-based species and fitness sharing, selection can kill them before weights adapt.

In [ ]:
d5 = ladder[-1]
x_d5, y_d5 = make_regression_task(d5)
no_species = neat_like_optimize(x_d5, y_d5, speciation=False, seed=123)
with_species = neat_like_optimize(x_d5, y_d5, speciation=True, seed=123)
no_species_hidden = no_species["best_genome"]["hidden"]
with_species_hidden = with_species["best_genome"]["hidden"]
print("no speciation fitness", round(no_species["best_fitness"], 4), "hidden", no_species_hidden)
print("with speciation fitness", round(with_species["best_fitness"], 4), "hidden", with_species_hidden)

## Evaluate it + Practice

- Metric: best fitness is negative regression error minus a small topology penalty.
- Baseline: compare with the initial linear genomes before add-node mutation.
- Sanity check: innovation distance should separate genomes with disjoint genes 2 and 3.
- Ablation: turn off speciation and track hidden-node survival.
- Failure signals: topology grows quickly but validation fitness does not improve.

Practice: raise `add_node_rate` and watch species counts.

Practice: change the topology penalty and inspect the best genome size.